<style>
@font-face {
  font-family: 'Roboto';
  font-style: normal;
  font-weight: 600;
  src: local('Roboto Semi-Bold'), local('Roboto-SemiBold'), url('../notebooks/assets/Roboto-SemiBold.ttf') format('truetype');
}
</style>

<div align="center">
  <img src="../notebooks/assets/Jupyter_AIKit_logo.svg" width="600">
</div>

<h2 style="color:#FFFFFF; text-align:center; font-family:'Roboto', sans-serif; font-size:24px; font-weight:600; letter-spacing:0.08em;">STEP-BY-STEP GUIDE TO RUN LLM MODELS ON-DEVICE (FOUNDLAB ATI INTEGRATED)</h2>

<h3 style="color:#44D62C; text-align:left;">Project Overview - FoundLab ATI Extension</h3>

AIKit is Razer's AI developer environment built to simplify and accelerate machine learning workflows on high-performance Razer hardware. 

This notebook includes the **FoundLab ATI (Algorithmic Trust Indicator)** extension, providing a zero-persistence cryptographic proof of the model's output in compliance with LGPD, the EU AI Act, and BCB 538.

<h3 style="color:#44D62C; text-align:left;">🚀 1. Run a Model with FoundLab ATI Middleware</h3>

We can start the vLLM server by explicitly injecting the FoundLab ATI Middleware. For this example, we mock the server process to show how the integration works behind the scenes.

In [ ]:
# Simulate running the vLLM server with FoundLab ATI Middleware injected.
# In production, the middleware is attached to the FastAPI application.

from foundlab_ati.generate_proof import generate_proof, verify_proof
import json

def mock_vllm_generate(prompt: str, model_name: str) -> dict:
    """Mock vLLM completion containing FoundLab ATI proof"""
    # 1. The model generates text
    output_text = "Quantum computing uses quantum mechanics to process information much faster than regular computers."
    
    # 2. The middleware automatically generates the ATI proof
    ati_proof = generate_proof(prompt, output_text, model_name)
    
    # 3. The response is returned to the client
    return {
        "id": "cmpl-mock123",
        "object": "text_completion",
        "created": 1710682000,
        "model": model_name,
        "choices": [
            {
                "text": output_text,
                "index": 0,
                "logprobs": None,
                "finish_reason": "stop"
            }
        ],
        "ati_proof": ati_proof
    }

<h3 style="color:#44D62C; text-align:left;">💬 2. Generate Text and Get ATI Proof</h3>

Once the model is running with the ATI Middleware, any prompt you send will return a secure response including the `"ati_proof"` object in the JSON body.

In [ ]:
prompt = "Explain quantum computing to a 12-year-old."
model = "Qwen/Qwen3-0.6B"

print(f"Sending prompt: '{prompt}'...")
response = mock_vllm_generate(prompt, model)

# Print the generated text
generated_text = response['choices'][0]['text']
print(f"\nResponse:\n{generated_text}\n")

# Print the ATI proof
print("=== Algorithmic Trust Indicator (ATI) ===")
print(json.dumps(response['ati_proof'], indent=2))

<h3 style="color:#44D62C; text-align:left;">✅ 3. Verify the Proof</h3>

You or a third-party auditor can mathematically verify that the output was indeed the answer to the specific input, without needing to save any persistent data on the device.

In [ ]:
# Extract the proof and verify it
proof = response['ati_proof']

is_valid = verify_proof(proof, prompt, generated_text)

if is_valid:
    print("✅ ATI Proof Verification: SUCCESS. The cryptographic signature matches the input and output.")
else:
    print("❌ ATI Proof Verification: FAILED. The data may have been tampered with.")